# Notebook 7: Citation Counts
We compare the total accumulated citations from publication to the last year with full citation data (i.e., 2025), focusing on the difference between *sharing* and *non-sharing* articles, with a drill-down to the sharing subcategories (granularities).
<br><br>
Our analyses have two steps: First, we use a uni-variate approach (t-test or Mann-Whitney U test, depending on normality). Second, we fit a multivariable regression to test whether the findings hold when controlling for other known predictors of article impact (e.g., publication age, number of authors, open-access status, etc.).
<br><br>
_Note: the bottom section of this notebook stores a now-deprecated analysis of citation residuals, which was used in the v1 manuscript but has been replaced by the multivariable regression above. It is kept here for reproducibility of the v1 results._

In [1]:
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from helpers.config import *
from helpers import dataset
from helpers.plotting import save_figure
from helpers.stats import compare_continuous, granularity_wald_test, model_diagnostics, pairwise_contrasts, sharing_effect

pio.renderers.default = "browser"

## Setup

In [2]:
combined, FEATURES_DF = dataset.load_or_build()
CITATIONS_DF = dataset.build_citations_frame(combined, FEATURES_DF)

CITATIONS_DF.columns

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\data_store (234 rows)


Index(['is_sharing_data', 'sharing_class', 'has_us_author', 'is_open_access',
       'has_preprint', 'log_weeks_since_pub', 'log_number_of_authors',
       'venue_impact', 'venue_name', 'log_citations'],
      dtype='str')

## Sharing vs. Non-Sharing

In [3]:
DV = "log_citations"
GROUPING_VARIABLE = "is_sharing_data"

### Univariate Comparison

In [4]:
# compare summary statistics of citation counts between sharing and non-sharing articles
CITATIONS_DF.groupby(GROUPING_VARIABLE)[DV].describe().T

is_sharing_data,0,1
count,154.000000,80.000000
mean,2.656499,2.707260
std,0.889737,0.938678
min,0.000000,0.693147
25%,2.079442,2.197225
50%,2.639057,2.673554
75%,3.295837,3.433987
max,4.700480,4.762174


In [5]:
# check for normality
sw_share = stats.shapiro(CITATIONS_DF[CITATIONS_DF[GROUPING_VARIABLE] == 1][DV])
print(f"Sharing Group is Normal:\t\t{'YES' if sw_share.pvalue > 0.05 else 'NO'} (W={sw_share.statistic :.3f}; p={sw_share.pvalue :.3f})")

sw_non_share = stats.shapiro(CITATIONS_DF[CITATIONS_DF[GROUPING_VARIABLE] == 0][DV])
print(f"Non-Sharing Group is Normal:\t{'YES' if sw_non_share.pvalue > 0.05 else 'NO'} (W={sw_non_share.statistic :.3f}; p={sw_non_share.pvalue :.3f})")

if sw_share.pvalue > 0.05 and sw_non_share.pvalue > 0.05:
    print("Both groups are normally distributed, using t-test.")
else:
    print("At least one group is not normally distributed, using Mann-Whitney U test.")

Sharing Group is Normal:		YES (W=0.980; p=0.228)
Non-Sharing Group is Normal:	YES (W=0.988; p=0.223)
Both groups are normally distributed, using t-test.


In [6]:
# run univariate test
citations_univariate_test_results, citations_univariate_summary = compare_continuous(
    data=CITATIONS_DF,
    share_feature=GROUPING_VARIABLE,
    tested_feature=DV,
    alternative="greater",
    verbose=False
)
citations_univariate_test_results = pd.Series(citations_univariate_test_results).rename(DV)
is_significant = citations_univariate_test_results.loc["p_val"] < 0.05

print(f"Univariate test is {'' if is_significant else 'NOT '}significant ({citations_univariate_test_results.loc['test']}={citations_univariate_test_results.loc['statistic']:.3f}, p={citations_univariate_test_results.loc['p_val']:.4f})")

citations_univariate_test_results

Univariate test is NOT significant (t=0.399, p=0.3451)


test                                                            t
alternative                                               greater
statistic                                                 0.39938
p_val                                                    0.345086
dof                                                     152.76833
effect_sizes    {'Cohen d': 0.05598459193658729, 'CLES': 0.515...
Name: log_citations, dtype: object

### Multivariable Regression
We compare *sharing* vs. *non-sharing* articles while controlling for other known predictors of citation impact. We use four different models, varying in the way they account for the venue of publication:
- Model 0 is an OLS regression with no venue term, with standard errors clustered by venue.
- Model 1 is an OLS regression, using the venue impact score as a predictor.
- Model 2 is a mixed-effects model (LME), with venue as a random effect.
- Model 3 merges the two approaches, using a LME with venue impact as a fixed effect and venue as a random effect. This is the most conservative model.

Models 0 and 3 **bracket** the sharing estimate rather than one superseding the other, because venue's causal role is not identified by these data (see Model 0 below).

#### Model 0 :: OLS without Venue
Model 0 omits the venue term entirely, so any sharing-venue-citation association is absorbed into the sharing coefficient rather than partialled out. Articles in the same venue are correlated, which a plain OLS ignores, so the standard errors are clustered by venue (with the small-sample correction and a *t* distribution on *G - 1* degrees of freedom, as there are only 23 venues).<br>
What the `is_sharing` coefficient *means* depends on a causal direction this corpus cannot settle:
- If sharing **leads to** publication in higher-impact venues, venue is a **mediator**, Model 0 estimates the total association, and Model 3 estimates the direct one.
- If instead a journal's data policy **drives** whether its authors share, venue is a **confounder**, Model 0 is confounded upward, and Model 3 is the estimate of interest.

The second reading is not far-fetched: sharing rates run from 0% (*Visual Cognition*, *Journal of Eye Movement Research*) to 82% (*PLoS ONE*), a spread that looks more like editorial policy than author choice. We therefore report Model 0 as a bound rather than as the preferred estimate, and do not interpret the gap between Models 0 and 3 as a mediated effect.
<br><br>
The model is specified as follows:
$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Status}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) $$

In [7]:
model0 = smf.ols(
    f"{DV} ~ C({GROUPING_VARIABLE}) + C(has_us_author) + C(is_open_access) + C(has_preprint) + log_weeks_since_pub + log_number_of_authors",
    data=CITATIONS_DF,
).fit(
    cov_type="cluster", cov_kwds={"groups": pd.factorize(CITATIONS_DF["venue_name"])[0], "use_correction": True}, use_t=True,
)

model0.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          log_citations   R-squared:                       0.144
Model:                            OLS   Adj. R-squared:                  0.121
Method:                 Least Squares   F-statistic:                     25.61
Date:                Wed, 14 Oct 2026   Prob (F-statistic):           7.26e-09
Time:                        09:29:11   Log-Likelihood:                -290.03
No. Observations:                 234   AIC:                             594.1
Df Residuals:                     227   BIC:                             618.3
Df Model:                           6                                         
Covariance Type:              cluster                                         
===========================================================================================
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                  -4.4799      0.955     -4.692      0.000      -6.460      -2.500
C(is_sharing_data)[T.1]     0.1370      0.095      1.447      0.162      -0.059       0.333
C(has_us_author)[T.1]       0.1974      0.091      2.180      0.040       0.010       0.385
C(is_open_access)[T.1]      0.3193      0.181      1.765      0.091      -0.056       0.694
C(has_preprint)[T.1]        0.1855      0.136      1.366      0.186      -0.096       0.467
log_weeks_since_pub         1.1377      0.172      6.616      0.000       0.781       1.494
log_number_of_authors       0.1467      0.139      1.058      0.301      -0.141       0.434
==============================================================================
Omnibus:                        4.672   Durbin-Watson:                   2.058
Prob(Omnibus):                  0.097   Jarque-Bera (JB):                4.515
Skew:                          -0.339   Prob(JB):                        0.105
Kurtosis:                       3.046   Cond. No.                         133.
==============================================================================

Notes:
[1] Standard Errors are robust to cluster correlation (cluster)
"""

#### Model 1 :: OLS with Venue Impact
Model 1 uses venue impact scores as a covariate, and does not account for the venue of publication as a random effect. The model is specified as follows:
$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Status}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) + \text{Impact Score} $$

In [8]:
model1 = smf.ols(
    f"{DV} ~ C({GROUPING_VARIABLE}) + C(has_us_author) + C(is_open_access) + C(has_preprint) + log_weeks_since_pub + log_number_of_authors + venue_impact",
    data=CITATIONS_DF,
    drop_cols=["sharing_class"]
).fit()

model1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          log_citations   R-squared:                       0.166
Model:                            OLS   Adj. R-squared:                  0.140
Method:                 Least Squares   F-statistic:                     6.416
Date:                Wed, 14 Oct 2026   Prob (F-statistic):           6.76e-07
Time:                        09:29:11   Log-Likelihood:                -286.98
No. Observations:                 234   AIC:                             590.0
Df Residuals:                     226   BIC:                             617.6
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
===========================================================================================
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                  -4.6207      1.195     -3.866      0.000      -6.976      -2.265
C(is_sharing_data)[T.1]     0.0752      0.127      0.592      0.555      -0.175       0.325
C(has_us_author)[T.1]       0.1801      0.112      1.612      0.108      -0.040       0.400
C(is_open_access)[T.1]      0.3515      0.157      2.232      0.027       0.041       0.662
C(has_preprint)[T.1]        0.1657      0.144      1.151      0.251      -0.118       0.449
log_weeks_since_pub         1.1266      0.199      5.675      0.000       0.735       1.518
log_number_of_authors       0.1066      0.132      0.810      0.419      -0.153       0.366
venue_impact                0.1099      0.045      2.443      0.015       0.021       0.199
==============================================================================
Omnibus:                        4.298   Durbin-Watson:                   2.072
Prob(Omnibus):                  0.117   Jarque-Bera (JB):                4.040
Skew:                          -0.317   Prob(JB):                        0.133
Kurtosis:                       3.108   Cond. No.                         144.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

#### Model 2 :: Mixed-Effects
Model 2 uses venue as a random effect, and does not include the venue impact score as a covariate. The model is specified as follows:
$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Status}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) + (1 | \text{Venue}) $$

In [9]:
model2 = smf.mixedlm(
    f"{DV} ~ C({GROUPING_VARIABLE}) + C(has_us_author) + C(is_open_access) + C(has_preprint) +  log_weeks_since_pub + log_number_of_authors",
    data=CITATIONS_DF,
    groups=CITATIONS_DF["venue_name"],
    drop_cols=["sharing_class"]
).fit(reml=True)

model2.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
              Mixed Linear Model Regression Results
==================================================================
Model:               MixedLM   Dependent Variable:   log_citations
No. Observations:    234       Method:               REML         
No. Groups:          23        Scale:                0.6646       
Min. group size:     1         Log-Likelihood:       -295.1953    
Max. group size:     43        Converged:            Yes          
Mean group size:     10.2                                         
------------------------------------------------------------------
                        Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------------
Intercept               -4.336    1.191 -3.642 0.000 -6.669 -2.002
C(is_sharing_data)[T.1]  0.086    0.133  0.646 0.518 -0.175  0.347
C(has_us_author)[T.1]    0.158    0.111  1.420 0.156 -0.060  0.376
C(is_open_access)[T.1]   0.339    0.177  1.913 0.056 -0.008  0.687
C(has_preprint)[T.1]     0.106    0.144  0.734 0.463 -0.177  0.389
log_weeks_since_pub      1.118    0.199  5.626 0.000  0.729  1.508
log_number_of_authors    0.150    0.131  1.141 0.254 -0.108  0.407
Group Var                0.068    0.068                           
==================================================================

"""

#### Model 3 :: Mixed-Effects with Venue Impact
Model 3 uses venue as a random effect, and includes the venue impact score as a covariate. The model is specified as follows:
$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Status}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) + \text{Impact Score} + (1 | \text{Venue}) $$

In [10]:
model3 = smf.mixedlm(
    f"{DV} ~ C({GROUPING_VARIABLE}) + C(has_us_author) + C(is_open_access) + C(has_preprint) +  log_weeks_since_pub + log_number_of_authors + venue_impact",
    data=CITATIONS_DF,
    groups=CITATIONS_DF["venue_name"],
    drop_cols=["sharing_class"]
).fit(reml=True)

model3.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
              Mixed Linear Model Regression Results
==================================================================
Model:               MixedLM   Dependent Variable:   log_citations
No. Observations:    234       Method:               REML         
No. Groups:          23        Scale:                0.6452       
Min. group size:     1         Log-Likelihood:       -293.4530    
Max. group size:     43        Converged:            Yes          
Mean group size:     10.2                                         
------------------------------------------------------------------
                        Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------------
Intercept               -4.711    1.184 -3.978 0.000 -7.033 -2.390
C(is_sharing_data)[T.1]  0.038    0.131  0.287 0.774 -0.220  0.295
C(has_us_author)[T.1]    0.138    0.110  1.260 0.208 -0.077  0.353
C(is_open_access)[T.1]   0.327    0.176  1.863 0.062 -0.017  0.672
C(has_preprint)[T.1]     0.098    0.142  0.693 0.488 -0.180  0.376
log_weeks_since_pub      1.118    0.196  5.699 0.000  0.733  1.502
log_number_of_authors    0.113    0.130  0.870 0.384 -0.142  0.369
venue_impact             0.180    0.073  2.472 0.013  0.037  0.323
Group Var                0.071    0.065                           
==================================================================

"""

### Model Selection
We have results from the 4 models above, and we can compare them to see how the inclusion of venue as a random effect and/or the venue impact score as a covariate affects the model convergence and predictive power.<br>
We also check if the conclusion - the estimated effect of sharing on citation counts - is consistent across the four models.

In [11]:
MODELS = {"0: OLS (no venue)": model0, "1: OLS": model1, "2: LME (venue)": model2, "3: LME (venue + impact)": model3}

In [12]:
# compare model diagnostics:
diagnostics = pd.DataFrame({name: model_diagnostics(res) for name, res in MODELS.items()}).T
diagnostics.index.name = "Model"

print(f"Best r^2 (marginal) is {diagnostics['R2 marginal'].max():.3f} in model {diagnostics['R2 marginal'].idxmax()}")
print(f"Best r^2 (conditional) is {diagnostics['R2 conditional'].max():.3f} in model {diagnostics['R2 conditional'].idxmax()}")
diagnostics

Best r^2 (marginal) is 0.183 in model 3: LME (venue + impact)
Best r^2 (conditional) is 0.264 in model 3: LME (venue + impact)


,converged,N,R2 marginal,R2 conditional,ICC
Model,,,,,
0: OLS (no venue),n/a,234,0.143757,0.143757,NaN
1: OLS,n/a,234,0.165779,0.165779,NaN
2: LME (venue),True,234,0.13077,0.211158,0.092482
3: LME (venue + impact),True,234,0.182732,0.264221,0.099709


In [13]:
# compare sharing effects across models
SHARING_TERM = "C(is_sharing_data)[T.1]"
sharing_effects = pd.DataFrame({name: sharing_effect(res, SHARING_TERM) for name, res in MODELS.items()}).T

same_sign = sharing_effects["b"].gt(0).all() or sharing_effects["b"].lt(0).all()
ci_includes_zero = (sharing_effects["CI low"] < 0).all() and (sharing_effects["CI high"] > 0).all()
print(f"Sharing coefficient ranges from {sharing_effects['b'].min():.3f} to {sharing_effects['b'].max():.3f} across models")
print(f"Same sign in all {len(MODELS)} models: {'YES' if same_sign else 'NO'}")
print(f"95% CI includes 0 in all {len(MODELS)} models: {'YES' if ci_includes_zero else 'NO'}")

sharing_effects

Sharing coefficient ranges from 0.038 to 0.137 across models
Same sign in all 4 models: YES
95% CI includes 0 in all 4 models: YES


,b,SE,p,CI low,CI high,% diff,% diff CI low,% diff CI high
0: OLS (no venue),0.136952,0.094643,0.161985,-0.059325,0.333230,14.677332,-5.759993,39.546790
1: OLS,0.075154,0.127001,0.554600,-0.175102,0.325411,7.805052,-16.062880,38.459947
2: LME (venue),0.086034,0.133169,0.518247,-0.174973,0.347041,8.984355,-16.052032,41.487518
3: LME (venue + impact),0.037662,0.131287,0.774212,-0.219655,0.294980,3.838050,-19.720446,34.309923


---

## Sharing Granularity

In [14]:
GROUPING_VARIABLE = "sharing_class"

CITATIONS_GRANULARITY_DF = CITATIONS_DF.loc[CITATIONS_DF[GROUPING_VARIABLE] != "NONE"]
CITATIONS_GRANULARITY_DF[GROUPING_VARIABLE] = CITATIONS_GRANULARITY_DF[GROUPING_VARIABLE].cat.remove_unused_categories()

# verify we are left only with the three sharing classes
CITATIONS_GRANULARITY_DF[GROUPING_VARIABLE].unique()

['FIXATION', 'TRIAL', 'PARTICIPANT']
Categories (3, str): ['PARTICIPANT' < 'TRIAL' < 'FIXATION']

### Univariate Comparison

In [15]:
# compare summary statistics of citation counts between sharing granularities
CITATIONS_GRANULARITY_DF.groupby(GROUPING_VARIABLE)[DV].describe().T

sharing_class,PARTICIPANT,TRIAL,FIXATION
count,23.000000,26.000000,31.000000
mean,2.714876,2.472695,2.898341
std,0.995560,0.927505,0.889432
min,0.693147,0.693147,1.098612
25%,2.191013,2.108887,2.302585
50%,2.995732,2.302585,2.944439
75%,3.312696,3.044522,3.637586
max,4.762174,4.127134,4.304065


In [16]:
all_normal = True
for sg in CITATIONS_GRANULARITY_DF[GROUPING_VARIABLE].unique():
    # check for normality
    sw = stats.shapiro(
        CITATIONS_GRANULARITY_DF[CITATIONS_GRANULARITY_DF[GROUPING_VARIABLE] == sg][DV]
    )
    sw_p = sw.pvalue
    print(f"{sg} Sharing Group is Normal:\t{'YES' if sw_p > 0.05 else 'NO'} (W={sw.statistic :.3f}; p={sw_p :.3f})")
    all_normal &= sw_p > 0.05

print()
print(f"All sharing groups are normally distributed: {'YES' if all_normal else 'NO'}")
print(f"Using {'ANOVA' if all_normal else 'Kruskal-Wallis'} test for group comparisons.")

FIXATION Sharing Group is Normal:	YES (W=0.960; p=0.285)
TRIAL Sharing Group is Normal:	YES (W=0.960; p=0.391)
PARTICIPANT Sharing Group is Normal:	YES (W=0.929; p=0.104)

All sharing groups are normally distributed: YES
Using ANOVA test for group comparisons.


In [17]:
# run univariate test
citations_granularity_results, _citations_granularity_summary = compare_continuous(
    data=CITATIONS_GRANULARITY_DF,
    share_feature=GROUPING_VARIABLE,
    tested_feature=DV,
    alternative="two-sided",
    verbose=False
)
citations_granularity_results = pd.Series(citations_granularity_results).rename(DV)
is_significant = citations_granularity_results.loc["p_val"] < 0.05

print(f"Univariate test is {'' if is_significant else 'NOT '}significant ({citations_granularity_results.loc['test']}={citations_granularity_results.loc['statistic']:.3f}, p={citations_granularity_results.loc['p_val']:.4f})")

citations_granularity_results

Univariate test is NOT significant (ANOVA=1.472, p=0.2358)


test                                    ANOVA
statistic                            1.472223
p_val                                  0.2358
dof                                   (2, 77)
effect_sizes    {'eta2': 0.03683115523541124}
Name: log_citations, dtype: object

### Multivariable Regression
We compare the three sharing granularities while controlling for other known predictors of citation impact.<br>
We use the same four models as above, varying in the way they account for the venue of publication:

#### Model 0 :: OLS without Venue (standard errors clustered by venue)
$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Granularity}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) $$

#### Model 1 :: OLS
$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Granularity}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) + \text{Impact Score} $$

#### Model 2 :: Mixed-Effects
$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Granularity}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) + (1 | \text{Venue}) $$

#### Model 3 :: Mixed-Effects with Venue Impact
$$ log(1 + \text{Raw Citation Counts}) \sim 1 + C(\text{Sharing Granularity}) + C(\text{Is OA}) + C(\text{Has US Author}) + C(\text{Has Preprint}) + log(\text{Weeks Since Pub.}) + log(\text{Number of Authors}) + \text{Impact Score} + (1 | \text{Venue}) $$

In [18]:
gran_model0 = smf.ols(
    f"{DV} ~ C({GROUPING_VARIABLE}) + C(has_us_author) + C(is_open_access) + C(has_preprint) + log_weeks_since_pub + log_number_of_authors",
    data=CITATIONS_GRANULARITY_DF,
).fit(
    cov_type="cluster", cov_kwds={"groups": pd.factorize(CITATIONS_GRANULARITY_DF["venue_name"])[0], "use_correction": True}, use_t=True,
)

gran_model0.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          log_citations   R-squared:                       0.263
Model:                            OLS   Adj. R-squared:                  0.191
Method:                 Least Squares   F-statistic:                     8.073
Date:                Wed, 14 Oct 2026   Prob (F-statistic):           0.000135
Time:                        09:29:18   Log-Likelihood:                -95.764
No. Observations:                  80   AIC:                             207.5
Df Residuals:                      72   BIC:                             226.6
Df Model:                           7                                         
Covariance Type:              cluster                                         
================================================================================================
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                       -5.4746      1.605     -3.411      0.003      -8.834      -2.116
C(sharing_class)[T.TRIAL]        0.1560      0.200      0.780      0.445      -0.262       0.574
C(sharing_class)[T.FIXATION]     0.2718      0.199      1.365      0.188      -0.145       0.689
C(has_us_author)[T.1]            0.4933      0.206      2.399      0.027       0.063       0.924
C(is_open_access)[T.1]           0.0903      0.542      0.167      0.869      -1.044       1.224
C(has_preprint)[T.1]            -0.2230      0.166     -1.347      0.194      -0.570       0.124
log_weeks_since_pub              1.3472      0.254      5.307      0.000       0.816       1.879
log_number_of_authors            0.1740      0.197      0.884      0.388      -0.238       0.586
==============================================================================
Omnibus:                        0.592   Durbin-Watson:                   2.015
Prob(Omnibus):                  0.744   Jarque-Bera (JB):                0.699
Skew:                          -0.058   Prob(JB):                        0.705
Kurtosis:                       2.557   Cond. No.                         127.
==============================================================================

Notes:
[1] Standard Errors are robust to cluster correlation (cluster)
"""

In [19]:
gran_model1 = smf.ols(
    f"{DV} ~ C({GROUPING_VARIABLE}) + C(has_us_author) + C(is_open_access) + C(has_preprint) + log_weeks_since_pub + log_number_of_authors + venue_impact",
    data=CITATIONS_GRANULARITY_DF,
    drop_cols=["is_sharing_data"]
).fit()

gran_model1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:          log_citations   R-squared:                       0.290
Model:                            OLS   Adj. R-squared:                  0.210
Method:                 Least Squares   F-statistic:                     3.624
Date:                Wed, 14 Oct 2026   Prob (F-statistic):            0.00138
Time:                        09:29:18   Log-Likelihood:                -94.252
No. Observations:                  80   AIC:                             206.5
Df Residuals:                      71   BIC:                             227.9
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
================================================================================================
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                       -5.7718      1.964     -2.939      0.004      -9.687      -1.856
C(sharing_class)[T.TRIAL]        0.1727      0.268      0.644      0.522      -0.362       0.707
C(sharing_class)[T.FIXATION]     0.3363      0.252      1.335      0.186      -0.166       0.839
C(has_us_author)[T.1]            0.4850      0.206      2.349      0.022       0.073       0.897
C(is_open_access)[T.1]           0.2122      0.367      0.578      0.565      -0.520       0.945
C(has_preprint)[T.1]            -0.2560      0.204     -1.256      0.213      -0.662       0.150
log_weeks_since_pub              1.3287      0.334      3.979      0.000       0.663       1.994
log_number_of_authors            0.0922      0.248      0.372      0.711      -0.402       0.586
venue_impact                     0.1369      0.083      1.654      0.103      -0.028       0.302
==============================================================================
Omnibus:                        0.278   Durbin-Watson:                   2.119
Prob(Omnibus):                  0.870   Jarque-Bera (JB):                0.453
Skew:                          -0.091   Prob(JB):                        0.797
Kurtosis:                       2.680   Cond. No.                         141.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [20]:
gran_model2 = smf.mixedlm(
    f"{DV} ~ C({GROUPING_VARIABLE}) + C(has_us_author) + C(is_open_access) + C(has_preprint) +  log_weeks_since_pub + log_number_of_authors",
    data=CITATIONS_GRANULARITY_DF,
    groups=CITATIONS_GRANULARITY_DF["venue_name"],
    drop_cols=["is_sharing_data"]
).fit(reml=True)

gran_model2.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
                 Mixed Linear Model Regression Results
=======================================================================
Model:                MixedLM     Dependent Variable:     log_citations
No. Observations:     80          Method:                 REML         
No. Groups:           20          Scale:                  0.6864       
Min. group size:      1           Log-Likelihood:         -100.5409    
Max. group size:      15          Converged:              Yes          
Mean group size:      4.0                                              
-----------------------------------------------------------------------
                             Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------------------
Intercept                    -5.211    1.997 -2.609 0.009 -9.126 -1.296
C(sharing_class)[T.TRIAL]     0.155    0.270  0.573 0.567 -0.375  0.684
C(sharing_class)[T.FIXATION]  0.276    0.255  1.081 0.280 -0.224  0.775
C(has_us_author)[T.1]         0.499    0.209  2.384 0.017  0.089  0.909
C(is_open_access)[T.1]        0.149    0.378  0.393 0.694 -0.593  0.890
C(has_preprint)[T.1]         -0.237    0.206 -1.153 0.249 -0.641  0.166
log_weeks_since_pub           1.294    0.344  3.764 0.000  0.620  1.968
log_number_of_authors         0.170    0.244  0.694 0.488 -0.309  0.649
Group Var                     0.029    0.068                           
=======================================================================

"""

In [21]:
gran_model3 = smf.mixedlm(
    f"{DV} ~ C({GROUPING_VARIABLE}) + C(has_us_author) + C(is_open_access) + C(has_preprint) +  log_weeks_since_pub + log_number_of_authors + venue_impact",
    data=CITATIONS_GRANULARITY_DF,
    groups=CITATIONS_GRANULARITY_DF["venue_name"],
    drop_cols=["is_sharing_data"]
).fit(reml=True)

gran_model3.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
                 Mixed Linear Model Regression Results
=======================================================================
Model:                MixedLM     Dependent Variable:     log_citations
No. Observations:     80          Method:                 REML         
No. Groups:           20          Scale:                  0.6808       
Min. group size:      1           Log-Likelihood:         -100.8948    
Max. group size:      15          Converged:              Yes          
Mean group size:      4.0                                              
-----------------------------------------------------------------------
                             Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------------------
Intercept                    -5.608    1.997 -2.809 0.005 -9.521 -1.695
C(sharing_class)[T.TRIAL]     0.166    0.269  0.618 0.536 -0.360  0.692
C(sharing_class)[T.FIXATION]  0.326    0.255  1.281 0.200 -0.173  0.825
C(has_us_author)[T.1]         0.487    0.207  2.352 0.019  0.081  0.893
C(is_open_access)[T.1]        0.235    0.374  0.628 0.530 -0.499  0.969
C(has_preprint)[T.1]         -0.260    0.204 -1.276 0.202 -0.660  0.140
log_weeks_since_pub           1.298    0.341  3.806 0.000  0.629  1.966
log_number_of_authors         0.099    0.247  0.399 0.690 -0.386  0.583
venue_impact                  0.134    0.087  1.549 0.121 -0.036  0.304
Group Var                     0.018    0.066                           
=======================================================================

"""

### Model Selection

In [22]:
GRAN_MODELS = {"0: OLS (no venue)": gran_model0, "1: OLS": gran_model1, "2: LME (venue)": gran_model2, "3: LME (venue + impact)": gran_model3}

In [23]:
# compare model diagnostics:
gran_diagnostics = pd.DataFrame({name: model_diagnostics(res) for name, res in GRAN_MODELS.items()}).T
gran_diagnostics.index.name = "Model"

print(f"Best r^2 (marginal) is {gran_diagnostics['R2 marginal'].max():.3f} in model {gran_diagnostics['R2 marginal'].idxmax()}")
print(f"Best r^2 (conditional) is {gran_diagnostics['R2 conditional'].max():.3f} in model {gran_diagnostics['R2 conditional'].idxmax()}")
gran_diagnostics

Best r^2 (marginal) is 0.290 in model 1: OLS
Best r^2 (conditional) is 0.290 in model 1: OLS


,converged,N,R2 marginal,R2 conditional,ICC
Model,,,,,
0: OLS (no venue),n/a,80,0.262608,0.262608,NaN
1: OLS,n/a,80,0.289957,0.289957,NaN
2: LME (venue),True,80,0.236478,0.267443,0.040556
3: LME (venue + impact),True,80,0.260466,0.27996,0.02636


In [24]:
# the two coefficients are each class vs. the PARTICIPANT baseline
GRAN_TERMS = ["C(sharing_class)[T.TRIAL]", "C(sharing_class)[T.FIXATION]"]
gran_sharing_effects = pd.DataFrame(
    {(name, term): sharing_effect(res, term) for name, res in GRAN_MODELS.items() for term in GRAN_TERMS}
).T
gran_sharing_effects.index.names = ["Model", "Term"]
gran_sharing_effects

b        SE  \
Model                   Term                                               
0: OLS (no venue)       C(sharing_class)[T.TRIAL]     0.155985  0.199881   
                        C(sharing_class)[T.FIXATION]  0.271771  0.199156   
1: OLS                  C(sharing_class)[T.TRIAL]     0.172703  0.268122   
                        C(sharing_class)[T.FIXATION]  0.336315  0.251960   
2: LME (venue)          C(sharing_class)[T.TRIAL]     0.154844  0.270208   
                        C(sharing_class)[T.FIXATION]  0.275528  0.254849   
3: LME (venue + impact) C(sharing_class)[T.TRIAL]     0.166043  0.268509   
                        C(sharing_class)[T.FIXATION]  0.326131  0.254585   

                                                             p    CI low  \
Model                   Term                                               
0: OLS (no venue)       C(sharing_class)[T.TRIAL]     0.444777 -0.262371   
                        C(sharing_class)[T.FIXATION]  0.188320 -0.145068   
1: OLS                  C(sharing_class)[T.TRIAL]     0.521573 -0.361918   
                        C(sharing_class)[T.FIXATION]  0.186207 -0.166078   
2: LME (venue)          C(sharing_class)[T.TRIAL]     0.566607 -0.374754   
                        C(sharing_class)[T.FIXATION]  0.279633 -0.223967   
3: LME (venue + impact) C(sharing_class)[T.TRIAL]     0.536319 -0.360225   
                        C(sharing_class)[T.FIXATION]  0.200183 -0.172846   

                                                       CI high     % diff  \
Model                   Term                                                
0: OLS (no venue)       C(sharing_class)[T.TRIAL]     0.574340  16.880814   
                        C(sharing_class)[T.FIXATION]  0.688610  31.228667   
1: OLS                  C(sharing_class)[T.TRIAL]     0.707324  18.851300   
                        C(sharing_class)[T.FIXATION]  0.838708  39.978002   
2: LME (venue)          C(sharing_class)[T.TRIAL]     0.684443  16.747639   
                        C(sharing_class)[T.FIXATION]  0.775023  31.722652   
3: LME (venue + impact) C(sharing_class)[T.TRIAL]     0.692310  18.062342   
                        C(sharing_class)[T.FIXATION]  0.825109  38.559722   

                                                      % diff CI low  \
Model                   Term                                          
0: OLS (no venue)       C(sharing_class)[T.TRIAL]        -23.077423   
                        C(sharing_class)[T.FIXATION]     -13.503666   
1: OLS                  C(sharing_class)[T.TRIAL]        -30.366046   
                        C(sharing_class)[T.FIXATION]     -15.301981   
2: LME (venue)          C(sharing_class)[T.TRIAL]        -31.254155   
                        C(sharing_class)[T.FIXATION]     -20.065824   
3: LME (venue + impact) C(sharing_class)[T.TRIAL]        -30.248051   
                        C(sharing_class)[T.FIXATION]     -15.873315   

                                                      % diff CI high  
Model                   Term                                          
0: OLS (no venue)       C(sharing_class)[T.TRIAL]          77.595776  
                        C(sharing_class)[T.FIXATION]       99.094717  
1: OLS                  C(sharing_class)[T.TRIAL]         102.855513  
                        C(sharing_class)[T.FIXATION]      131.337654  
2: LME (venue)          C(sharing_class)[T.TRIAL]          98.266692  
                        C(sharing_class)[T.FIXATION]      117.064313  
3: LME (venue + impact) C(sharing_class)[T.TRIAL]          99.832649  
                        C(sharing_class)[T.FIXATION]      128.212922

### Pairwise Contrasts
Comparing all three granularities against each other: each contrast is the second level minus the first (`TRIAL vs. PARTICIPANT` is the TRIAL coefficient, `FIXATION vs. TRIAL` is the difference between the two coefficients). `t` is a z statistic in the mixed-effects models. `p_FDR` is Benjamini-Hochberg across the three contrasts of each model. `ratio` is `exp(b)`, the ratio of the geometric means, and `% change` is `ratio - 1`.

In [25]:
gran_contrasts = pd.concat(
    {name: pairwise_contrasts(res, GRAN_TERMS, baseline="PARTICIPANT") for name, res in GRAN_MODELS.items()},
    names=["Model"],
)
gran_contrasts

b        SE  \
Model                   Contrast                                       
0: OLS (no venue)       TRIAL vs. PARTICIPANT     0.155985  0.199881   
                        FIXATION vs. PARTICIPANT  0.271771  0.199156   
                        FIXATION vs. TRIAL        0.115787  0.243104   
1: OLS                  TRIAL vs. PARTICIPANT     0.172703  0.268122   
                        FIXATION vs. PARTICIPANT  0.336315  0.251960   
                        FIXATION vs. TRIAL        0.163612  0.245338   
2: LME (venue)          TRIAL vs. PARTICIPANT     0.154844  0.270208   
                        FIXATION vs. PARTICIPANT  0.275528  0.254849   
                        FIXATION vs. TRIAL        0.120684  0.247288   
3: LME (venue + impact) TRIAL vs. PARTICIPANT     0.166043  0.268509   
                        FIXATION vs. PARTICIPANT  0.326131  0.254585   
                        FIXATION vs. TRIAL        0.160089  0.245780   

                                                         t         p  \
Model                   Contrast                                       
0: OLS (no venue)       TRIAL vs. PARTICIPANT     0.780388  0.444777   
                        FIXATION vs. PARTICIPANT  1.364611  0.188320   
                        FIXATION vs. TRIAL        0.476285  0.639302   
1: OLS                  TRIAL vs. PARTICIPANT     0.644120  0.521573   
                        FIXATION vs. PARTICIPANT  1.334798  0.186207   
                        FIXATION vs. TRIAL        0.666884  0.507008   
2: LME (venue)          TRIAL vs. PARTICIPANT     0.573056  0.566607   
                        FIXATION vs. PARTICIPANT  1.081143  0.279633   
                        FIXATION vs. TRIAL        0.488031  0.625528   
3: LME (venue + impact) TRIAL vs. PARTICIPANT     0.618388  0.536319   
                        FIXATION vs. PARTICIPANT  1.281030  0.200183   
                        FIXATION vs. TRIAL        0.651349  0.514821   

                                                     p_FDR     ratio  \
Model                   Contrast                                       
0: OLS (no venue)       TRIAL vs. PARTICIPANT     0.639302  1.168808   
                        FIXATION vs. PARTICIPANT  0.564960  1.312287   
                        FIXATION vs. TRIAL        0.639302  1.122756   
1: OLS                  TRIAL vs. PARTICIPANT     0.521573  1.188513   
                        FIXATION vs. PARTICIPANT  0.521573  1.399780   
                        FIXATION vs. TRIAL        0.521573  1.177757   
2: LME (venue)          TRIAL vs. PARTICIPANT     0.625528  1.167476   
                        FIXATION vs. PARTICIPANT  0.625528  1.317227   
                        FIXATION vs. TRIAL        0.625528  1.128268   
3: LME (venue + impact) TRIAL vs. PARTICIPANT     0.536319  1.180623   
                        FIXATION vs. PARTICIPANT  0.536319  1.385597   
                        FIXATION vs. TRIAL        0.536319  1.173615   

                                                   % change  
Model                   Contrast                             
0: OLS (no venue)       TRIAL vs. PARTICIPANT     16.880814  
                        FIXATION vs. PARTICIPANT  31.228667  
                        FIXATION vs. TRIAL        12.275627  
1: OLS                  TRIAL vs. PARTICIPANT     18.851300  
                        FIXATION vs. PARTICIPANT  39.978002  
                        FIXATION vs. TRIAL        17.775743  
2: LME (venue)          TRIAL vs. PARTICIPANT     16.747639  
                        FIXATION vs. PARTICIPANT  31.722652  
                        FIXATION vs. TRIAL        12.826823  
3: LME (venue + impact) TRIAL vs. PARTICIPANT     18.062342  
                        FIXATION vs. PARTICIPANT  38.559722  
                        FIXATION vs. TRIAL        17.361488

#### Omnibus Test
Running a Wald test to determine if the granularity coefficients differ from zero, jointly. This is a more stringent test than the per-coefficient or per-contrast results reported above, because it tests the null hypothesis that all granularity coefficients are zero simultaneously, rather than testing each coefficient individually.

In [26]:
gran_omnibus = pd.DataFrame({name: granularity_wald_test(res, GRAN_TERMS) for name, res in GRAN_MODELS.items()}).T
n_significant = int((gran_omnibus["p"] < 0.05).sum())
print(f"Omnibus p-value ranges from {gran_omnibus['p'].min():.3f} to {gran_omnibus['p'].max():.3f} across models")
print(f"Significant (p < 0.05) in {n_significant} of {len(gran_omnibus)} models")

gran_omnibus

Omnibus p-value ranges from 0.376 to 0.557 across models
Significant (p < 0.05) in 0 of 4 models


,test,statistic,p,df_denom,df_num
0: OLS (no venue),F,1.029527,0.376262,19,2
1: OLS,F,0.897887,0.412005,71,2
2: LME (venue),chi2,1.170168,0.557059,2,None
3: LME (venue + impact),chi2,1.655147,0.437109,2,None


In [27]:
# TODO: drop this to see the (deprecated) residuals analysis
1/0

ZeroDivisionError: division by zero

---

## Age-Adjusted Citation Residuals (SUPERSEDED)

**This section is superseded and does not appear in the revised manuscript.** It is the analysis
reported in the submitted version, kept so the published v1 results stay reproducible. Everything
from this heading to the end of the notebook belongs to it, including the per-group regressions at
the bottom.

It regresses log-citations on log-age alone, then compares the residuals between sharing and
non-sharing articles. The multivariable model above replaces it, for two reasons. Age is only one
of several predictors of citation count, so residualizing on age alone leaves venue, preprint
status and the rest uncontrolled. And the two-stage form (fit, then test the residuals) gives the
right point estimate with the wrong standard errors, because the second stage treats fitted
residuals as though they were observed data.

These cells deliberately use `Pub2UpdateTime` for article age rather than
`weeks_through_shared_year`. That is what v1 was computed from, and changing it here would stop
this section reproducing the published numbers, which is its only remaining purpose.

### Citation Counts over Time
#### (1) Quantify the Benefit of Data Sharing
We want to quantify the citation benefit of data sharing over time.<br>
To do so we first fit a log-log regression model to all data points, predicting citation counts from time since publication: $$\log(\text{Citations} +1) = \beta_0 + \beta_1 \log(\text{Weeks Since Publication}) + \epsilon$$(we add $1$ to the citation count to avoid $\log(0)$)<br><br>
Then, we compute the residuals (i.e., the difference between observed and predicted citation counts) for each article, and compare the residuals between sharing and non-sharing articles and across types of data shared.

In [ ]:
# fit log-linear model to all data
x_log = np.log(combined["Pub2UpdateTime"] / pd.Timedelta(weeks=1))
y_log = np.log(combined['TotalCitations'] + 1)  # add 1 to avoid log(0)
slope, intercept = np.polyfit(x_log, y_log, deg=1)

# compute residuals
ylog_pred = intercept + slope * x_log
residuals = (y_log - ylog_pred).rename("citation_residuals")
combined_with_residuals = pd.concat([combined, residuals], axis=1)

# extract residual by share/not-share
non_sharing_residuals = combined_with_residuals.loc[
    ~combined_with_residuals["is_sharing_data"], "citation_residuals"
]
sharing_residuals = combined_with_residuals.loc[
    combined_with_residuals["is_sharing_data"], "citation_residuals"
]

In [ ]:
# check for normality of residuals
print("Are citation residuals normally distributed for non-sharing articles?")
residuals_non_sharing_shapiro = stats.shapiro(non_sharing_residuals)
print(f"{"YES" if residuals_non_sharing_shapiro.pvalue > 0.05 else "NO"} (W={residuals_non_sharing_shapiro.statistic:.3f}, p={residuals_non_sharing_shapiro.pvalue:.4f})\n")

print("Are citation residuals normally distributed for sharing articles?")
residuals_sharing_shapiro = stats.shapiro(sharing_residuals)
print(f"{"YES" if residuals_sharing_shapiro.pvalue > 0.05 else "NO"} (W={residuals_sharing_shapiro.statistic:.3f}, p={residuals_sharing_shapiro.pvalue:.4f})\n")

# run Levene's test for homogeneity of variance
levene_test = stats.levene(non_sharing_residuals, sharing_residuals)
is_equal_var = levene_test.pvalue > 0.05
print(f"Levene's test for homogeneity of variance: W={levene_test.statistic:.3f}, p={levene_test.pvalue:.4f}")
print(f"Sample have {'EQUAL' if is_equal_var else 'UNEQUAL'} variances.")

In [ ]:
# run two-sample t-test
residuals_ttest = pg.ttest(
    non_sharing_residuals, sharing_residuals, paired=False, alternative="less", correction=~is_equal_var,
)
print("t-test:")
display(residuals_ttest)

# run Mann-Whitney U-test
residuals_mwtest = pg.mwu(non_sharing_residuals, sharing_residuals, alternative="less")
print("MW U-test:")
display(residuals_mwtest)

# calculate the percentage difference in mean citations
non_share_residuals_mean = non_sharing_residuals.mean()
share_residuals_mean = sharing_residuals.mean()
citation_ratio = np.exp(share_residuals_mean - non_share_residuals_mean)
percentage_diff = (citation_ratio - 1) * 100
print(f"Data-Sharing has {percentage_diff:.1f}% citations compared with Non-Sharing.")

#### Is there a Difference Between Types of Data Shared?

In [ ]:
share_group_residuals = []
for share_group in ["FIXATION", "TRIAL", "PARTICIPANT"]:
    group_residuals = (
        combined_with_residuals
        .loc[combined_with_residuals["data_sharing_class"] == share_group, "citation_residuals"]
        .dropna()
        .reset_index(drop=True)
        .rename(share_group)
    )
    share_group_residuals.append(group_residuals)
    shapiro_result = stats.shapiro(group_residuals)
    print(f"Are citation-residuals (in log-log space) normally distributed for {share_group} sharing articles?")
    print(f"{"YES" if shapiro_result.pvalue > 0.05 else "NO"} (W={shapiro_result.statistic:.3f}, p={shapiro_result.pvalue:.4f})\n")

print("Statistical Tests:")
# run one-way anova
print("One-Way ANOVA")
sharegroup_residuals_anova = stats.f_oneway(*share_group_residuals)
display(sharegroup_residuals_anova)

# run Kruskal-Wallis
print("Kruskal-Wallis")
sharegroup_residuals_kruskal = stats.kruskal(*share_group_residuals)
display(sharegroup_residuals_kruskal)

#### Visualizing Citation Residuals

In [ ]:
column_titles = ["Citations for Time Since Publication", "Citation Residuals by Data Sharing"]
residuals_fig = make_subplots(
    rows=1, cols=2, shared_xaxes=False, shared_yaxes=False,
    column_titles=column_titles, column_widths=[0.7, 0.3],
    horizontal_spacing=0.075,
)

# single-article data
for is_share in sorted(combined_with_residuals["is_sharing_data"].unique()):
    is_sharing_subset = combined_with_residuals[combined_with_residuals["is_sharing_data"] == is_share]
    legendgroup = "Sharing" if is_share else "Not Sharing"
    color = CLASS_COLORS[legendgroup.upper()]
    # left: scatter plot of citations over time
    for share_class in is_sharing_subset["data_sharing_class"].unique():
        label = share_class.title() if is_share else "Not Sharing"
        symbol = {"FIXATION": "circle", "TRIAL": "triangle-up", "PARTICIPANT": "square", "NONE": "x", }[share_class]
        class_subset = is_sharing_subset.loc[combined_with_residuals["data_sharing_class"] == share_class]
        residuals_fig.add_trace(
            row=1, col=1,
            trace=go.Scatter(
                name=label, legendgroup=label,
                x=class_subset["Pub2UpdateTime"] / pd.Timedelta(weeks=1),
                y=class_subset["TotalCitations"],
                mode="markers",
                marker=dict(color=CLASS_COLORS[share_class], symbol=symbol, size=8, ),
            )
        )
    residuals_fig.update_xaxes(
        row=1, col=1, title=dict(text="Weeks Since Publication", font=AXIS_TITLE_FONT, standoff=10), tickfont=AXIS_TICK_FONT,
    )
    residuals_fig.update_yaxes(
        row=1, col=1, title=dict(text="Total Citations", font=AXIS_TITLE_FONT, standoff=5), tickfont=AXIS_TICK_FONT,
    )

    # right: violin plot of residuals
    label = "Sharing" if is_share else "Not Sharing"
    residuals_fig.add_trace(
        row=1, col=2,
        trace=go.Violin(
            x0=0, y=is_sharing_subset["citation_residuals"],  # keeping it in log-space for visualization & statistics
            name=label, legendgroup=label, scalegroup=1,
            side='positive' if is_share else 'negative',
            fillcolor=color, line_color=color,
            width=0.95, spanmode="hard",
            points=False, pointpos=0, jitter=0.5,
            box=dict(visible=False, width=0.5, line=dict(color="black")),
            meanline=dict(visible=False, color='black'),
            showlegend=False,
        )
    )
    residuals_fig.update_xaxes(
        row=1, col=2, title=None, showticklabels=False,
    )
    residuals_fig.update_yaxes(
        row=1, col=2,
        title=dict(text="log-Residual Citations", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
    )
    # add annotations
    residuals_fig.add_annotation(
        row=1, col=2,
        x=0.2 if is_share else -0.2,
        xanchor="left" if is_share else "right", xref="x",
        y=1.75, yanchor="top", yref="y2",
        text=f"{label.replace(" ", "<br>")}",
        font={**AXIS_TITLE_FONT, "color": color},
        showarrow=False,
    )
    residuals_fig.add_annotation(   # p value annotation
        row=1, col=2,
        x=0.125, xanchor="left", xref="x",
        y=-2.5, yanchor="middle", yref="y2",
        text=f"<i>p={residuals_ttest["p_val"].iloc[0] :.3f}</i>",
        font=AXIS_TITLE_FONT,
        showarrow=False,
    )

# add log-log trendline for citations over time
x_logspace = np.linspace(x_log.min(), x_log.max(), 100)
y_logpred = intercept + slope * x_logspace
y_pred = np.exp(y_logpred) - 1  # invert log transform
x_orig = np.exp(x_logspace)
residuals_fig.add_trace(
    row=1, col=1,
    trace=go.Scatter(
        name="Expected<br>Citations",
        x=x_orig, y=y_pred,
        mode="lines",
        line=dict(color='black', width=2.5, dash='dash'),
    )
)

# update layout and show
for ann in residuals_fig.layout.annotations:
    if ann.text not in column_titles:
        continue
    x = -0.03 if ann.text == column_titles[0] else 0.67
    ann.update(dict(
        font=AXIS_TITLE_FONT,
        x=x, xanchor="left", xref="paper",
        y=1.1, yanchor="top", yref="paper",
))
residuals_fig.update_xaxes(zeroline=False,)
residuals_fig.update_yaxes(
    showgrid=True, gridcolor='lightgrey', gridwidth=1.5,
    zeroline=False,
)
residuals_fig.update_xaxes(
    row=1, col=1, range=[140, 505]
)
residuals_fig.update_yaxes(
    row=1, col=1, range=[-5, 119]
)
residuals_fig.update_layout(
    width=1000, height=400,
    title=dict(
        text="<b>Estimating Number of Citations by Weeks Past Publication</b>",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.99, yanchor="top"
    ),
    violinmode='overlay',
    violingap=0,
    legend=dict(
        visible=True,
        orientation="v",
        entrywidth=30,
        bgcolor='rgba(255,255,255,0.2)',
        borderwidth=2, bordercolor='rgba(0,0,0,1)',
        x=0.01, xanchor="left", xref="paper",
        y=1.0, yanchor="top", yref="paper",
        font=LEGEND_FONT, font_size=10,
    ),
    margin=dict(t=60, b=5, l=50, r=10, pad=0),
    template="plotly_white"
)
residuals_fig.show()

In [ ]:
# flip to True to export this figure into `output/`
if False:
    save_figure(residuals_fig, "citation_residuals_by_data_sharing.png", width=1000, height=400)

#### (2) Regress each Sharing Group Separately
exploratory analysis: fit separate log-log regression models for each sharing group

In [ ]:
citation_counts_fig = go.Figure()
for group in combined["data_sharing_class"].unique():
    subset = combined[combined["data_sharing_class"] == group]
    color = CLASS_COLORS[group]
    symbol = 'circle' if group != "NONE" else 'x'
    citation_counts_fig.add_trace(go.Scatter(
        name=group, legendgroup=group,
        x=subset["Pub2UpdateTime"] / pd.Timedelta(weeks=1), y=subset["TotalCitations"],
        mode="markers", marker=dict(color=color, size=8, symbol=symbol),
    ))
    # fit log-log trendline per group
    x_log = np.log(subset["Pub2UpdateTime"] / pd.Timedelta(weeks=1))
    y_log = np.log(subset['TotalCitations'] + 1)  # add 1 to avoid log(0)
    slope, intercept = np.polyfit(x_log, y_log, deg=1)
    x_logspace = np.linspace(x_log.min(), x_log.max(), 100)
    y_logpred = intercept + slope * x_logspace
    y_pred = np.exp(y_logpred) - 1  # invert log transform
    x_orig = np.exp(x_logspace)
    citation_counts_fig.add_trace(go.Scatter(
        name=f"Trendline ({group})",
        x=x_orig, y=y_pred, mode="lines", line=dict(color=color, width=2, dash='dot'),
        showlegend=False,
    ))
    print("Group: {0: <12}".format(group) + f"Slope: {slope:.3f},\tIntercept: {intercept:.2f}")

# add general log-log trendline
x_log = np.log(combined["Pub2UpdateTime"] / pd.Timedelta(weeks=1))
y_log = np.log(combined['TotalCitations'] + 1)
slope, intercept = np.polyfit(x_log, y_log, deg=1)
x_logspace = np.linspace(x_log.min(), x_log.max(), 100)
y_logpred = intercept + slope * x_logspace
y_pred = np.exp(y_logpred) - 1
x_orig = np.exp(x_logspace)
citation_counts_fig.add_trace(go.Scatter(
    name="Trendline (all data)",
    x=x_orig, y=y_pred, mode="lines", line=dict(color='black', width=2, dash='dash'),
))
print("{0: <19}".format("All Data:") + f"Slope: {slope:.3f},\tIntercept: {intercept:.2f}")

citation_counts_fig.update_layout(
    width=800, height=300,
    title=dict(
        text="Citation Counts over Time Since Publication", font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top",
    ),
    xaxis=dict(
        title=dict(text="Weeks Since Publication", font=AXIS_TITLE_FONT, standoff=10),
        tickfont=AXIS_TICK_FONT,
        # type="log",
        zeroline=False,
    ),
    yaxis=dict(
        title=dict(text="Total Citation Counts", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
        # type="log",
        zeroline=True,
    ),
    legend=dict(
        orientation="h", bgcolor='rgba(0,0,0,0)',
        x=0.5, xanchor="center", y=0.96, yanchor="bottom",
    ),
    margin=dict(t=50, b=10, l=10, r=10, pad=0),
)
citation_counts_fig